[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_06_deep_learning/ch30_cnns_computer_vision/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 30 Walkthrough: Convolutional Neural Networks (CNNs) & Medical Image Classification

**Book Reference:** Chapter 30, Sections 30.1 – 30.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every experiment runs 100% offline on deterministic synthetic single-channel medical lesion patches (mimicking chest X-ray / microscopy nodules) in under 3 seconds on CPU.

In [ ]:
# @title Step 0: Click Play (▶) to Load PyTorch, NumPy & Compare MLP vs CNN
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import sharukh_tutor as tutor

# Why flattened MLPs explode on medical images (e.g., 256x256 X-ray):
mlp_first_layer = nn.Linear(256 * 256, 256)
conv_first_layer = nn.Conv2d(1, 8, kernel_size=3, padding=1)
mlp_params = sum(p.numel() for p in mlp_first_layer.parameters())
conv_params = sum(p.numel() for p in conv_first_layer.parameters())
print(f"PyTorch {torch.__version__} | Device: cpu")
print(f"MLP layer-1 params (256x256 -> 256): {mlp_params:,}")
print(f"Conv2d(1->8, 3x3) params (any HxW):  {conv_params:,}")
ratio = mlp_params / conv_params
print(f"Parameter reduction:                 {ratio:,.0f}x")


## Section 30.3.1 — Pure-NumPy 2D Cross-Correlation Verified Against `F.conv2d`

In [ ]:
def conv2d_numpy(image_2d, kernel_2d, stride=1, padding=0):
    img = np.asarray(image_2d, dtype=np.float32)
    ker = np.asarray(kernel_2d, dtype=np.float32)
    if padding > 0:
        img = np.pad(
            img, ((padding, padding), (padding, padding)), mode="constant"
        )
    kh, kw = ker.shape
    h_out = (img.shape[0] - kh) // stride + 1
    w_out = (img.shape[1] - kw) // stride + 1
    out = np.zeros((h_out, w_out), dtype=np.float32)
    for i in range(h_out):
        for j in range(w_out):
            patch = img[
                i * stride : i * stride + kh,
                j * stride : j * stride + kw,
            ]
            out[i, j] = np.sum(patch * ker)
    return out

rng = np.random.default_rng(42)
sample_img = rng.normal(size=(6, 6)).astype(np.float32)
sobel_x = np.array(
    [[1.0, 0.0, -1.0], [2.0, 0.0, -2.0], [1.0, 0.0, -1.0]],
    dtype=np.float32,
)
np_out = conv2d_numpy(sample_img, sobel_x, stride=1, padding=1)
pt_out = F.conv2d(
    torch.from_numpy(sample_img).unsqueeze(0).unsqueeze(0),
    torch.from_numpy(sobel_x).unsqueeze(0).unsqueeze(0),
    stride=1,
    padding=1,
).squeeze().numpy()
max_err = float(np.max(np.abs(np_out - pt_out)))
print(f"Output shape: {np_out.shape} | Max diff vs F.conv2d: {max_err:.2e}")


## Section 30.3.2 — Output-Shape Arithmetic & `MaxPool2d` vs. `AdaptiveAvgPool2d`

In [ ]:
def conv2d_shape(h_in, w_in, kernel_size, stride=1, padding=0):
    if stride <= 0:
        raise ValueError("stride must be positive")
    h_out = (h_in + 2 * padding - kernel_size) // stride + 1
    w_out = (w_in + 2 * padding - kernel_size) // stride + 1
    if h_out <= 0 or w_out <= 0:
        raise ValueError("output dimensions must be positive")
    return (int(h_out), int(w_out))

for label, args in [
    ("Valid 3x3 (16x16)", (16, 16, 3, 1, 0)),
    ("Same 3x3  (16x16)", (16, 16, 3, 1, 1)),
    ("Stride-2  (32x24)", (32, 24, 5, 2, 2)),
]:
    print(f"{label} -> {conv2d_shape(*args)}")

x_16 = torch.randn(2, 8, 16, 16)
x_24 = torch.randn(2, 8, 24, 24)
mp = nn.MaxPool2d(kernel_size=2)
gap = nn.AdaptiveAvgPool2d(1)
print(f"MaxPool2d(2) on 16x16: {tuple(mp(x_16).shape)}")
print(f"MaxPool2d(2) on 24x24: {tuple(mp(x_24).shape)}")
print(f"AdaptiveAvgPool2d(1) on 16x16: {tuple(gap(x_16).shape)}")
print(f"AdaptiveAvgPool2d(1) on 24x24: {tuple(gap(x_24).shape)}")


## Section 30.3.3 — `MedicalPatchCNN` & `ResidualConvBlock` Skip Connections

In [ ]:
class MedicalPatchCNN(nn.Module):
    """Compact 2D CNN for single-channel medical patch classification."""

    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        return self.fc(self.features(x))

class ResidualConvBlock(nn.Module):
    """Two-layer Conv+BN residual block with identity skip connection."""

    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)

demo_cnn = MedicalPatchCNN(num_classes=1)
res_blk = ResidualConvBlock(channels=8)
total_p = sum(p.numel() for p in demo_cnn.parameters())
print(f"MedicalPatchCNN total params: {total_p:,}")
s16 = tuple(demo_cnn(torch.zeros(4, 1, 16, 16)).shape)
s24 = tuple(demo_cnn(torch.zeros(2, 1, 24, 24)).shape)
print(f"Output on 16x16: {s16} | on 24x24: {s24}")
res_in = torch.randn(3, 8, 16, 16)
print(
    f"ResidualConvBlock: {tuple(res_in.shape)} -> "
    f"{tuple(res_blk(res_in).shape)}"
)


## Section 30.3.4 — Synthetic Medical Lesion Patches & Training `MedicalPatchCNN`

In [ ]:
def generate_synthetic_lesion_dataset(n_samples=160, img_size=16, seed=42):
    rng = np.random.default_rng(seed)
    n_neg = n_samples // 2
    n_pos = n_samples - n_neg
    labels = np.array([0] * n_neg + [1] * n_pos, dtype=np.float32)
    rng.shuffle(labels)
    yy, xx = np.meshgrid(
        np.arange(img_size), np.arange(img_size), indexing="ij"
    )
    images = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    for i, lbl in enumerate(labels):
        base = rng.uniform(0.15, 0.30)
        patch = rng.normal(base, 0.12, size=(img_size, img_size))
        if lbl == 1.0:
            cy = rng.integers(3, img_size - 3)
            cx = rng.integers(3, img_size - 3)
            amp = rng.uniform(0.95, 1.25)
            dist_sq = (yy - cy) ** 2 + (xx - cx) ** 2
            patch = patch + amp * np.exp(-dist_sq / (2.0 * 1.5 ** 2))
        images[i, 0] = np.clip(patch, 0.0, 2.0)
    return torch.from_numpy(images), torch.from_numpy(labels)

X_all, y_all = generate_synthetic_lesion_dataset(200, img_size=16, seed=42)
idx = np.arange(len(y_all))
tr_i, va_i = train_test_split(
    idx, test_size=0.25, stratify=y_all.numpy(), random_state=42
)
X_tr, y_tr = X_all[tr_i], y_all[tr_i]
X_va, y_va = X_all[va_i], y_all[va_i]

torch.manual_seed(42)
cnn = MedicalPatchCNN(num_classes=1)
opt = torch.optim.Adam(cnn.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1, 16):
    cnn.train()
    perm = torch.randperm(len(X_tr))
    ep_loss = 0.0
    for start in range(0, len(X_tr), 32):
        b = perm[start : start + 32]
        opt.zero_grad()
        loss = loss_fn(cnn(X_tr[b]).squeeze(-1), y_tr[b])
        loss.backward()
        opt.step()
        ep_loss += loss.item() * len(b)
    ep_loss /= len(X_tr)
    if epoch in (1, 5, 10, 15):
        print(f"Epoch {epoch:2d} | Train BCE Loss: {ep_loss:.5f}")

cnn.eval()
with torch.no_grad():
    va_probs = torch.sigmoid(cnn(X_va).squeeze(-1)).numpy()
va_pred = (va_probs >= 0.5).astype(np.float32)
va_acc = float((va_pred == y_va.numpy()).mean())
va_auc = float(roc_auc_score(y_va.numpy(), va_probs))
print(f"Val Accuracy: {va_acc:.4f} | Val ROC-AUC: {va_auc:.4f}")


## Section 30.3.5 — Transfer Learning: Freezing the Convolutional Backbone

In [ ]:
def freeze_backbone(model, head_name="fc"):
    prefix = f"{head_name}."
    trainable, frozen = 0, 0
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(prefix)
        if p.requires_grad:
            trainable += p.numel()
        else:
            frozen += p.numel()
    return {"trainable_params": trainable, "frozen_params": frozen}

transfer_net = MedicalPatchCNN(num_classes=1)
transfer_net.load_state_dict(cnn.state_dict())
counts = freeze_backbone(transfer_net, head_name="fc")
w_conv0 = transfer_net.features[0].weight.detach().clone()
w_fc0 = transfer_net.fc.weight.detach().clone()

head_opt = torch.optim.Adam(
    [p for p in transfer_net.parameters() if p.requires_grad], lr=0.05
)
head_opt.zero_grad()
loss_fn(transfer_net(X_tr[:16]).squeeze(-1), y_tr[:16]).backward()
head_opt.step()

conv_changed = not torch.equal(w_conv0, transfer_net.features[0].weight)
fc_changed = not torch.equal(w_fc0, transfer_net.fc.weight)
print(f"Transfer counts: {counts}")
print(f"Conv1 changed? {conv_changed} | Head fc changed? {fc_changed}")


## Section 30.3.6 — Input-Gradient Saliency Map: Locating the Lesion Pixel

In [ ]:
pos_idx = int(np.where(y_va.numpy() == 1.0)[0][0])
x_single = X_va[pos_idx : pos_idx + 1]

cnn.eval()
cnn.zero_grad()
x_grad = x_single.clone().detach().requires_grad_(True)
logit = cnn(x_grad).squeeze()
logit.backward()

saliency = x_grad.grad.detach().abs().squeeze().cpu().numpy()
sal_peak = tuple(
    int(v) for v in np.unravel_index(int(np.argmax(saliency)), (16, 16))
)
patch_2d = x_single.squeeze().numpy()
true_peak = tuple(
    int(v) for v in np.unravel_index(int(np.argmax(patch_2d)), (16, 16))
)
dy = sal_peak[0] - true_peak[0]
dx = sal_peak[1] - true_peak[1]
dist_px = float(np.hypot(dy, dx))
prob = float(torch.sigmoid(logit.detach()).item())
print(f"Predicted lesion probability: {prob:.4f}")
print(f"Saliency peak: {sal_peak} | Brightest lesion pixel: {true_peak}")
print(f"Localization offset: {dist_px:.2f} px")


## Section 30.3.7 — Break It On Purpose! (CNN Error Anatomy)

In [ ]:
# Intentional Bug #1: Passing 3-channel RGB tensor into 1-channel Conv2d
gray_conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
rgb_batch = torch.zeros(2, 3, 16, 16)
gray_conv(rgb_batch)


In [ ]:
# Intentional Bug #2: Passing 2D (H, W) array without batch/channel dims
gray_conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
flat_2d = torch.zeros(16, 16)
gray_conv(flat_2d)


In [ ]:
# Intentional Bug #3: Calling backward() when ALL parameters are frozen
frozen_net = MedicalPatchCNN(num_classes=1)
for p in frozen_net.parameters():
    p.requires_grad = False
out = frozen_net(torch.zeros(2, 1, 16, 16)).squeeze(-1)
nn.BCEWithLogitsLoss()(out, torch.ones(2)).backward()


## Section 30.4 — Stage 4 Blueprint: End-to-End Medical Patch CNN Audit Pipeline

In [ ]:
def run_lesion_cnn_audit(n_samples=200, epochs=15, seed=42):
    """Audit shapes, train MedicalPatchCNN & locate lesion via saliency."""
    X, y = generate_synthetic_lesion_dataset(n_samples, 16, seed)
    idx = np.arange(len(y))
    tr, va = train_test_split(
        idx, test_size=0.25, stratify=y.numpy(), random_state=seed
    )
    torch.manual_seed(seed)
    model = MedicalPatchCNN(num_classes=1)
    shapes, handles = [], []

    def _make_hook(layer_name):
        def _hook(_mod, _inp, out):
            shapes.append((layer_name, tuple(out.shape)))
        return _hook

    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            handles.append(mod.register_forward_hook(_make_hook(name)))
    try:
        model.eval()
        with torch.no_grad():
            model(torch.zeros(4, 1, 16, 16))
    finally:
        for h in handles:
            h.remove()

    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(tr))
        for s in range(0, len(tr), 32):
            b = tr[perm[s : s + 32].numpy()]
            opt.zero_grad()
            loss = loss_fn(model(X[b]).squeeze(-1), y[b])
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X[va]).squeeze(-1)).numpy()
    val_auc = round(float(roc_auc_score(y[va].numpy(), probs)), 4)
    pos_i = int(np.where(y[va].numpy() == 1.0)[0][0])
    x_one = X[va][pos_i : pos_i + 1].clone().detach().requires_grad_(True)
    model.zero_grad()
    logit = model(x_one).squeeze()
    logit.backward()
    sal = x_one.grad.detach().abs().squeeze().cpu().numpy()
    peak_rc = np.unravel_index(int(np.argmax(sal)), sal.shape)
    peak = (int(peak_rc[0]), int(peak_rc[1]))
    return {"shapes": shapes, "val_auc": val_auc, "peak_coord": peak}

rep = run_lesion_cnn_audit(n_samples=200, epochs=15, seed=42)
for layer_name, shp in rep["shapes"]:
    print(f"  {layer_name:10s} -> {shp}")
print(
    f"Val ROC-AUC: {rep['val_auc']:.4f} | "
    f"Saliency peak: {rep['peak_coord']}"
)
